# 02. Wiring an Azure AI Foundry Project to an MCP Knowledge-Base Connection

This notebook doesn't process any document at all — it's the **infrastructure wiring step** that sits between the raw document extraction in `01_document_intelligence.ipynb` and the agent that will later reason over invoices. It registers a "remote tool" **connection** on an Azure AI Foundry project, pointing at a Model Context Protocol (MCP) endpoint exposed by an Azure AI Search **knowledge base** (an agentic-retrieval preview capability). That connection is a prerequisite that `03_cloudxeus_invoice_agent.ipynb` references later by name (`project_connection_id`) so the agent's MCP tool can actually reach the CloudXeus product knowledge base at runtime.

**Difficulty: Intermediate** (this is an ARM *management-plane* REST call with its own auth token scope, not a typical data-plane SDK call).

## Prerequisites

**Python packages**
```bash
pip install azure-identity requests python-dotenv
```
Both `azure-identity` and `requests` are already present in the repo root `requirements.txt`.

**Azure resources**
- An **Azure AI Foundry project** (same project used throughout `11_azure_ai_foundry/`).
- An **Azure AI Search** service with an **agentic retrieval knowledge base** already configured (preview capability — this is *not* a plain search index, it's the newer "knowledge base" resource exposing an MCP endpoint at `/knowledgebases/<name>/mcp`).
- The identity running this notebook (`az login` account, via `DefaultAzureCredential`) needs write access (e.g. Contributor) on the Foundry project's Azure resource, since this call goes through the Azure Resource Manager (ARM) control plane, not a data-plane API key.

**Environment variables** (add to the root `.env`):

```
AZURE_AI_PROJECT_RESOURCE_ID=/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<foundry-account>/projects/<project-name>
AZURE_AI_PROJECT_MCP_CONNECTION_NAME=cloudxeus-product-kb-mcp-connection
AZURE_SEARCH_SERVICE_ENDPOINT=https://<your-search-service>.search.windows.net
AZURE_SEARCH_KNOWLEDGE_BASE_NAME=kb-cloudxeus-course-products
```

`AZURE_AI_PROJECT_RESOURCE_ID` is the full ARM resource ID of your Foundry project — to find your own, use the same `az rest` lookup documented in `11_azure_ai_foundry/README.md`:
```bash
az rest --method get --url "https://management.azure.com/subscriptions/<sub>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<account>/projects?api-version=2025-04-01-preview"
```

## What You'll Learn

- The difference between Azure's **control plane** (ARM, `management.azure.com`) and **data plane** (the actual AI service endpoints) — this script talks to the control plane
- How to mint a bearer token scoped to a specific audience with `get_bearer_token_provider`
- How Azure AI Foundry project **connections** model external tool/data endpoints, including the `RemoteTool` category used for MCP servers
- Why the connection uses `authType: "ProjectManagedIdentity"` instead of a static API key
- How this connection is later consumed by name via `MCPTool(project_connection_id=...)` in the agent-creation step

### Imports, credential, and environment-driven configuration

We replace every hardcoded ID/name from the original script with `os.getenv(...)` lookups. `DefaultAzureCredential` picks up your `az login` session (or managed identity, environment variables, etc. — whichever is available) with no code changes needed between local dev and a deployed environment.

In [ ]:
import os

import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv

load_dotenv()

credential = DefaultAzureCredential()

PROJECT_RESOURCE_ID = os.getenv("AZURE_AI_PROJECT_RESOURCE_ID")
PROJECT_CONNECTION_NAME = os.getenv(
    "AZURE_AI_PROJECT_MCP_CONNECTION_NAME", "cloudxeus-product-kb-mcp-connection"
)

SEARCH_SERVICE_ENDPOINT = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
KNOWLEDGE_BASE_NAME = os.getenv("AZURE_SEARCH_KNOWLEDGE_BASE_NAME", "kb-cloudxeus-course-products")

mcp_endpoint = (
    f"{SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    "?api-version=2026-05-01-preview"
)


### Getting an ARM-scoped bearer token

Because we're calling the Azure Resource Manager control-plane API directly (via `requests.put`, not an SDK method), we need to mint our own bearer token rather than let an SDK client handle auth transparently. `get_bearer_token_provider` wraps `DefaultAzureCredential` into a callable that returns a fresh token string scoped to the `https://management.azure.com/.default` audience — that's the audience ARM itself expects, distinct from the `audience` field we'll set *inside* the connection payload below (which tells Foundry what audience to request when *it* later calls Azure AI Search on the agent's behalf).

💡 **Exam tip (AI-102):** Azure AI Foundry (and Cognitive Services generally) supports both key-based and Entra ID (Azure AD) authentication. Scoped bearer tokens via `DefaultAzureCredential` + `get_bearer_token_provider` are the recommended pattern for service-to-service calls in production, since there's no static secret to rotate or leak.

🔄 **Alternatives:** `az rest --method put --url ... --body ...` from the Azure CLI accomplishes the same ARM call without writing any Python — useful for one-off provisioning scripts or CI/CD pipelines.

In [ ]:
bearer_token_provider = get_bearer_token_provider(
    credential,
    "https://management.azure.com/.default",
)

headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
    "Content-Type": "application/json",
}


### Creating (or updating) the project connection

A `PUT` to the ARM connections endpoint is idempotent — running this notebook again with the same `PROJECT_CONNECTION_NAME` updates the existing connection rather than erroring. The connection's `category` is `"RemoteTool"` (an MCP-style tool endpoint, as opposed to, say, a data-source connection for RAG grounding), its `target` is the knowledge base's MCP URL, and `authType: "ProjectManagedIdentity"` means Foundry authenticates to Azure AI Search using the **project's own managed identity** at call time — no API key stored anywhere.

💡 **Exam tip (AI-102):** Foundry project *connections* are the general mechanism for wiring external resources (search indexes, storage, other APIs, and — as here — MCP tool servers) into a project so agents/tools can reference them by name instead of embedding credentials in agent definitions. Expect the exam to test that you understand connections as the "credential and endpoint" indirection layer for grounding data and tools.

🔄 **Alternatives:** You could provision this same connection declaratively with a Bicep/ARM template or Terraform's `azapi` provider as part of infrastructure-as-code, rather than an imperative Python/REST call — more appropriate once this moves out of a learning notebook into a real deployment pipeline.

In [ ]:
response = requests.put(
    f"https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}"
    "?api-version=2025-10-01-preview",
    headers=headers,
    json={
        "name": PROJECT_CONNECTION_NAME,
        "type": "Microsoft.MachineLearningServices/workspaces/connections",
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {
                "ApiType": "Azure"
            },
        },
    },
)

print(response.status_code)
print(response.text)

response.raise_for_status()

print(f"Connection '{PROJECT_CONNECTION_NAME}' created or updated successfully.")


## Summary

We minted an ARM-scoped bearer token and issued a `PUT` to the Azure Resource Manager to create (or update) a Foundry project connection named `cloudxeus-product-kb-mcp-connection`. That connection now points at an Azure AI Search knowledge base's MCP endpoint and is configured to authenticate via the project's managed identity. Nothing was extracted or analyzed in this notebook — its only output is a `2xx` HTTP response confirming the connection exists. The next notebook, `03_cloudxeus_invoice_agent.ipynb`, references this exact connection name when it builds the agent's `MCPTool`.

## Try It Yourself

1. **Easy:** Run the notebook twice in a row and inspect `response.status_code` both times — confirm it behaves idempotently (create vs. update) rather than erroring on the second run.
2. **Intermediate:** Add a `GET` request (`https://management.azure.com{PROJECT_RESOURCE_ID}/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview`) after the `PUT` to fetch back and pretty-print the connection you just created, confirming the `target` and `authType` were stored as expected.
3. **Advanced:** Parametrize this notebook so it can create connections for *multiple* knowledge bases (e.g. one per product line) by looping over a list of `(connection_name, knowledge_base_name)` pairs, and print a summary table of which connections succeeded.